# Voice-Controlled Data Assistant

In [5]:
!pip cache purge --quiet

In [6]:
!pip install aiofiles==24.1.0 \
             av==15.1.0 \
             langchain==0.3.27 \
             langchain-community==0.3.25 \
             langchain-core==0.3.76 \
             langchain-text-splitters==0.3.11 \
             langsmith==0.3.45 \
             requests-toolbelt==1.0.0 \
             langchain-openai==0.3.33 \
             tiktoken==0.8.0 \
             langchain-singlestore==1.0.0 \
             livekit==1.0.16 \
             livekit-agents==1.2.14 \
             livekit-api==1.0.6 \
             livekit-blingfire==1.0.0 \
             livekit-plugins-openai==1.2.14 \
             livekit-plugins-silero==1.2.14 \
             livekit-protocol==1.0.8 \
             onnxruntime==1.23.1 \
             openai==2.2.0 \
             --no-deps --quiet

In [7]:
%%writefile agent_core.py

import asyncio
import logging
import os
import warnings

from langchain_community.agent_toolkits import create_sql_agent
from langchain_community.utilities import SQLDatabase
from langchain_openai import ChatOpenAI
from livekit.agents import Agent, AgentSession, JobContext, RoomInputOptions, function_tool
from livekit.plugins import openai, silero
from sqlalchemy.exc import SAWarning

warnings.filterwarnings("ignore", category = DeprecationWarning)
warnings.filterwarnings("ignore", category = SAWarning)

logger = logging.getLogger("ai-agent-core")
logger.setLevel(logging.INFO)
logging.getLogger("sqlalchemy").setLevel(logging.ERROR)

def initialize_langchain():
    """Initialize LangChain SQL agent with SingleStore database."""
    connection_url = os.environ.get("SINGLESTOREDB_URL")
    if not connection_url:
        raise EnvironmentError("SINGLESTOREDB_URL environment variable is required")

    langchain_llm = ChatOpenAI(
        model = "gpt-4o-mini",
        temperature = 0
    )

    db = SQLDatabase.from_uri(
        connection_url,
        include_tables = ["tick"]
    )

    sql_agent = create_sql_agent(
        llm = langchain_llm,
        db = db,
        agent_type = "openai-tools",
        verbose = False
    )

    logger.info("LangChain SQL agent initialized successfully")
    return sql_agent

def clean_response_for_voice(text):
    """Clean database output for natural voice delivery."""
    # Remove special characters and formatting
    cleaned = text.replace('|', ' ').replace('```', '').replace('---', '').replace('**', '').replace('*', '')

    # Join non-empty lines
    lines = [line.strip() for line in cleaned.splitlines() if line.strip()]
    result = " ".join(lines)

    # Limit length
    max_length = 800
    if len(result) > max_length:
        result = result[:max_length] + "... and more."

    return result

class Assistant(Agent):
    """LiveKit voice agent that exposes a LangChain SQL agent as a callable tool."""

    def __init__(self, sql_agent) -> None:
        if not os.environ.get("OPENAI_API_KEY"):
            raise EnvironmentError("OPENAI_API_KEY environment variable is required")

        self._sql_agent = sql_agent

        super().__init__(
            instructions = (
                "You are a voice assistant that answers questions about stock tick data. "
                "Always call the query_stock_data tool to answer questions about the "
                "database, never guess at numbers yourself. Keep spoken answers short "
                "and conversational since they will be read aloud. Never use markdown "
                "formatting such as asterisks, bullet points or code blocks - respond "
                "in plain spoken sentences only."
            ),
        )

        logger.info("Voice assistant initialized with LangChain SQL tool")

    @function_tool()
    async def query_stock_data(self, question: str) -> str:
        """Look up an answer to a question about stock tick data by querying the
        SingleStore database.

        Args:
            question: The user's natural language question about the tick data.
        """
        logger.info(f"Processing query: {question}")

        # Query LangChain SQL agent (runs synchronously in thread pool)
        try:
            result = await asyncio.get_event_loop().run_in_executor(
                None,
                lambda: self._sql_agent.invoke({"input": question})
            )

            # Extract and clean response
            output = result.get("output", "No results found") if isinstance(result, dict) else str(result)
            response_text = clean_response_for_voice(output)

            logger.info(f"LangChain response: {response_text}")
            return response_text

        except Exception as e:
            logger.error(f"Query error: {e}", exc_info = True)
            return "I encountered an error querying the database."

async def entrypoint(ctx: JobContext):
    """Main entrypoint for LiveKit agent worker."""
    logger.info(f"Job started for room: {ctx.room.name}")

    # Initialize LangChain SQL agent
    sql_agent = initialize_langchain()

    # Connect to LiveKit room
    await ctx.connect()
    logger.info(f"Connected to room: {ctx.room.name}")

    # Build the agent session - a real LLM drives the conversation and calls
    # query_stock_data as a tool, so the full STT -> LLM -> TTS loop stays intact
    session = AgentSession(
        stt = openai.STT(model = "whisper-1"),
        llm = openai.LLM(model = "gpt-4o-mini"),
        tts = openai.TTS(model = "gpt-4o-mini-tts"),
        vad = silero.VAD.load(),
    )

    disconnected_future = asyncio.get_event_loop().create_future()

    @ctx.room.on("disconnected")
    def on_disconnected():
        logger.info("Room disconnected")
        if not disconnected_future.done():
            disconnected_future.set_result(True)

    # ctx.room "disconnected" only fires once the whole room is torn down by the
    # server, which can take up to the departure timeout after the caller leaves.
    # Watching for the caller's participant leaving lets us end the job immediately
    # instead of waiting that out.
    @ctx.room.on("participant_disconnected")
    def on_participant_disconnected(participant):
        if not ctx.room.remote_participants and not disconnected_future.done():
            logger.info(f"Caller {participant.identity} left - ending session")
            disconnected_future.set_result(True)

    await session.start(
        room = ctx.room,
        agent = Assistant(sql_agent),
        room_input_options = RoomInputOptions(),
    )
    logger.info("Agent running - queries routed through the query_stock_data tool")

    # Greet the caller so they know the assistant is ready
    await session.generate_reply(
        instructions = "Greet the user and let them know they can ask about stock tick data."
    )

    await disconnected_future

    # Clean up our own connection rather than waiting for the server to do it
    try:
        await ctx.room.disconnect()
    except Exception:
        pass  # already disconnected, e.g. if the room itself was torn down

    logger.info("Session finished")

Writing agent_core.py


In [8]:
import asyncio
import logging
import os
import warnings

from agent_core import entrypoint
from livekit.agents import WorkerOptions, Worker
from singlestoredb.management import get_secret
from urllib.parse import quote

In [9]:
logger = logging.getLogger("ai-agent")
logger.setLevel(logging.INFO)

logging.getLogger("httpx").setLevel(logging.ERROR)
logging.getLogger("httpcore").setLevel(logging.ERROR)
logging.getLogger("livekit").setLevel(logging.WARNING)
logging.getLogger("sqlalchemy").setLevel(logging.ERROR)

<div class="alert alert-block alert-warning">
    <b class="fa fa-solid fa-exclamation-circle"></b>
    <div>
        <p><b>Action Required</b></p>
        <p>Select the database from the drop-down menu at the top of this notebook. It updates the <b>connection_url</b> which is used by SQLAlchemy to make connections to the selected database.</p>
    </div>
</div>

In [10]:
from sqlalchemy import *

db_connection = create_engine(connection_url)
url = db_connection.url

host = url.host
port = url.port
database = url.database
username = "admin"
password = get_secret("password")

# Validate before building the connection string
missing = [
    name for name, value in [
        ("host", host),
        ("port", port),
        ("database", database),
        ("username", username),
        ("password", password),
    ]
    if not value
]

if missing:
    raise ValueError(
        f"Missing required SingleStore connection detail(s): {', '.join(missing)}. "
        "Check that a database is selected in the notebook's connection pulldown "
        "before running this cell."
    )

SINGLESTOREDB_URL = f"singlestoredb://{username}:{quote(password, safe = '')}@{quote(host, safe = '')}:{port}/{database}"

In [11]:
LIVEKIT_URL = get_secret("LIVEKIT_URL")
LIVEKIT_API_KEY = get_secret("LIVEKIT_API_KEY")
LIVEKIT_API_SECRET = get_secret("LIVEKIT_API_SECRET")

os.environ["LIVEKIT_URL"] = LIVEKIT_URL
os.environ["LIVEKIT_API_KEY"] = LIVEKIT_API_KEY
os.environ["LIVEKIT_API_SECRET"] = LIVEKIT_API_SECRET
os.environ["OPENAI_API_KEY"] = get_secret("OPENAI_API_KEY")
os.environ["SINGLESTOREDB_URL"] = SINGLESTOREDB_URL

<div class="alert alert-block alert-info">
    <b class="fa fa-solid fa-info-circle"></b>
    <div>
        <p><b>Connecting a Browser Tab</b></p>
        <p>Rather than generating a join token ourselves, we'll use LiveKit Cloud's hosted <b>token server</b>. From the LiveKit Cloud dashboard, select this project, open <b>Settings</b> in the left navigation and switch on the <b>Token server</b> toggle. A sandbox ID such as <code>token-server-xxxxxx</code> appears below the toggle once it's on - copy it. Open the accompanying <code>voice-assistant.html</code> page in any browser tab (no local install, no build step), paste in the sandbox ID and click <b>Start Call</b>. The page fetches a join token directly from LiveKit Cloud and creates a fresh room each time it connects, which the worker below picks up automatically.</p>
    </div>
</div>

In [13]:
async def main_worker():
    logger.info("Starting LiveKit Worker with LangChain SQL integration...")
    opts = WorkerOptions(
        entrypoint_fnc = entrypoint,
        ws_url = LIVEKIT_URL,
        api_key = LIVEKIT_API_KEY,
        api_secret = LIVEKIT_API_SECRET,
    )

    worker = Worker(opts)
    worker_task = asyncio.create_task(worker.run())

    try:
        await worker_task
    except asyncio.CancelledError:
        pass
    finally:
        logger.info("Worker execution stopped.")

In [14]:
# Execute this cell to start the worker, then open voice-assistant.html in a browser tab
await main_worker()

INFO:ai-agent:Starting LiveKit Worker with LangChain SQL integration...
INFO:ai-agent-core:Job started for room: voice-controlled-gbv4c5cd
INFO:ai-agent-core:LangChain SQL agent initialized successfully
INFO:ai-agent-core:Connected to room: voice-controlled-gbv4c5cd
INFO:ai-agent-core:Voice assistant initialized with LangChain SQL tool
INFO:ai-agent-core:Agent running - queries routed through the query_stock_data tool
INFO:ai-agent-core:Processing query: How many rows are in the tick table?
INFO:ai-agent-core:LangChain response: There are 379,764 rows in the tick table.
INFO:ai-agent-core:Processing query: Show me the last two ticks for symbol BBRQ-FX.
INFO:ai-agent-core:LangChain response: The last two ticks for the symbol BBRQ-FX are as follows: 1. Date: November 24, 2015 - Open: 999.83 - High: 1017.41 - Low: 985.60 - Close: 1004.39 - Volume: 2,066,982 2. Date: November 23, 2015 - Open: 1020.74 - High: 1039.59 - Low: 1010.85 - Close: 1019.04 - Volume: 2,005,610
INFO:ai-agent-core:Pro